# CSV for Data Engineering

## From API JSON Response to CSV Files

This notebook uses small data engineering examples.

In many beginner data engineering projects, we often follow this flow:

```text
API
 ↓
JSON response
 ↓
Flatten into rows and columns
 ↓
Save as CSV
 ↓
Read CSV for SQL, dashboard, analysis, or machine learning
```

By the end of this notebook, you should be able to:

1. Explain what a CSV file is.
2. Understand rows, columns, headers, delimiters, and missing values.
3. Convert API-like JSON data into a table.
4. Save data as CSV using pandas.
5. Read CSV files using pandas.
6. Use Python's built-in `csv` library for basic CSV operations.
7. Understand common CSV issues such as commas inside text, encoding, missing values, and data types.
8. Explain when CSV is useful and when formats such as Parquet may be better.

## 1. What is CSV?

CSV stands for **Comma-Separated Values**.

A CSV file is a simple text file that stores tabular data.

Example:

```csv
region,taxi_count,weather
Central,120,Cloudy
East,85,Rain
West,95,Fair
```

A CSV file usually has:

| Concept | Meaning |
|---|---|
| Header | The first row containing column names |
| Row | One record |
| Column | One field or attribute |
| Delimiter | The character separating columns, usually comma |
| Cell | One value in the table |

CSV is simple, readable, and supported by almost every data tool.

## 2. Why CSV Matters in Data Engineering

CSV is common because it is:

| Advantage | Explanation |
|---|---|
| Simple | Easy to create and inspect |
| Portable | Supported by many tools |
| Human-readable | Can be opened in text editors and spreadsheets |
| Useful for small/medium data | Good for teaching and quick analysis |

However, CSV also has limitations:

| Limitation | Explanation |
|---|---|
| No strong schema | Data types are not preserved well |
| Larger file size | Usually larger than compressed columnar formats |
| Slower analytical queries | Not optimized for column-based analytics |
| Ambiguous formatting | Commas, quotes, encoding, dates can cause issues |

In real data engineering, CSV is useful, but Parquet is often better for large analytical datasets.

In [1]:
import json
import csv
import pandas as pd
from pprint import pprint
from pathlib import Path

DATA_FILE = Path("taxi_weather_simple.csv")
if not DATA_FILE.exists():
    for parent in Path.cwd().resolve().parents:
        candidate = parent / "02_clean_transform" / "taxi_weather_simple.csv"
        if candidate.exists():
            DATA_FILE = candidate
            break

MISSING_VALUES_FILE = Path("missing_values_example.csv")
DATA_TYPE_FILE = Path("data_type_example.csv")
COMMA_INSIDE_TEXT_FILE = Path("comma_inside_text.csv")


## 3. A Simple CSV Example

This notebook uses `taxi_weather_simple.csv` as the main example file.

The next cell creates the same small table, so the notebook can still regenerate the CSV when needed.


In [2]:
data = [
    {"region": "Central", "taxi_count": 120, "weather": "Cloudy"},
    {"region": "East", "taxi_count": 85, "weather": "Rain"},
    {"region": "West", "taxi_count": 95, "weather": "Fair"}
]

df = pd.DataFrame(data)
df

,region,taxi_count,weather
0,Central,120,Cloudy
1,East,85,Rain
2,West,95,Fair


In [3]:
df.to_csv(DATA_FILE, index=False)

print(f"Saved {DATA_FILE}")


Saved taxi_weather_simple.csv


### Note

If we do not use `index=False`, pandas will save the DataFrame index as an extra column.

In [4]:
df_read = pd.read_csv(DATA_FILE)
df_read


,region,taxi_count,weather
0,Central,120,Cloudy
1,East,85,Rain
2,West,95,Fair


## 4. CSV as Plain Text

A CSV file is just text.

Let us open the file and print its content.

In [5]:
with open(DATA_FILE, "r", encoding="utf-8") as f:
    content = f.read()

print(content)


region,taxi_count,weather
Central,120,Cloudy
East,85,Rain
West,95,Fair



## 5. Common pandas CSV Functions

pandas is the most convenient library for CSV operations in data analysis and data engineering lessons.

| Function | Purpose |
|---|---|
| `pd.read_csv()` | Read CSV into a DataFrame |
| `df.to_csv()` | Save DataFrame to CSV |
| `df.head()` | Preview first rows |
| `df.info()` | Check columns and data types |
| `df.describe()` | Summary statistics |
| `df.isna()` | Check missing values |
| `df.dropna()` | Drop rows with missing values |
| `df.fillna()` | Fill missing values |

In [6]:
df = pd.read_csv(DATA_FILE)

df.head()


,region,taxi_count,weather
0,Central,120,Cloudy
1,East,85,Rain
2,West,95,Fair


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   region      3 non-null      object
 1   taxi_count  3 non-null      int64 
 2   weather     3 non-null      object
dtypes: int64(1), object(2)
memory usage: 204.0+ bytes


In [8]:
df.describe()

,taxi_count
count,3.000000
mean,100.000000
std,18.027756
min,85.000000
25%,90.000000
50%,95.000000
75%,107.500000
max,120.000000


## 6. Selecting Columns from CSV Data

After reading a CSV file, we can select columns like a normal DataFrame.

In [9]:
df[["region", "taxi_count", "weather"]]


,region,taxi_count,weather
0,Central,120,Cloudy
1,East,85,Rain
2,West,95,Fair


## 7. Filtering Rows from CSV Data

Example: select rows where `taxi_count` is greater than 90.


In [10]:
df[df["taxi_count"] > 90]


,region,taxi_count,weather
0,Central,120,Cloudy
2,West,95,Fair


## 8. Missing Values in CSV

CSV files often contain missing values.

Example:

```csv
region,taxi_count,weather
Central,120,Cloudy
East,,Rain
West,95,
```

When pandas reads this file, empty cells usually become `NaN`.

In [11]:
csv_text = '''region,taxi_count,weather
Central,120,Cloudy
East,,Rain
West,95,
'''

with open(MISSING_VALUES_FILE, "w", encoding="utf-8") as f:
    f.write(csv_text)

df_missing = pd.read_csv(MISSING_VALUES_FILE)
df_missing

,region,taxi_count,weather
0,Central,120.0,Cloudy
1,East,NaN,Rain
2,West,95.0,NaN


In [12]:
df_missing.isna()

,region,taxi_count,weather
0,False,False,False
1,False,True,False
2,False,False,True


In [13]:
df_missing.isna().sum()

region        0
taxi_count    1
weather       1
dtype: int64

### Handling Missing Values

Common options:

| Method | Meaning |
|---|---|
| `dropna()` | Remove rows with missing values |
| `fillna()` | Fill missing values |
| Manual inspection | Understand why values are missing |

The correct method depends on the business context.

In [14]:
# Fill missing taxi_count with 0
# Fill missing weather with "Unknown"

df_filled = df_missing.fillna({
    "taxi_count": 0,
    "weather": "Unknown"
})

df_filled

,region,taxi_count,weather
0,Central,120.0,Cloudy
1,East,0.0,Rain
2,West,95.0,Unknown


## 9. Data Types in CSV

CSV files are plain text, so data types may not always be read correctly.

For example:

- ID values may be read as numbers.
- Dates may be read as strings.
- Missing values may change integer columns into floats.
- Leading zeros may disappear.

Let us look at an example.

In [15]:
csv_text = '''student_id,postal_code,score,date
001,010001,85,2026-05-26
002,020002,90,2026-05-27
003,030003,,2026-05-28
'''

with open(DATA_TYPE_FILE, "w", encoding="utf-8") as f:
    f.write(csv_text)

df_types = pd.read_csv(DATA_TYPE_FILE)
df_types

,student_id,postal_code,score,date
0,1,10001,85.0,2026-05-26
1,2,20002,90.0,2026-05-27
2,3,30003,NaN,2026-05-28


In [16]:
df_types.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   student_id   3 non-null      int64  
 1   postal_code  3 non-null      int64  
 2   score        2 non-null      float64
 3   date         3 non-null      object 
dtypes: float64(1), int64(2), object(1)
memory usage: 228.0+ bytes


Notice that some ID-like fields may lose leading zeros if pandas treats them as numbers.

We can control data types using the `dtype` argument.

In [17]:
df_types_fixed = pd.read_csv(
    DATA_TYPE_FILE,
    dtype={
        "student_id": str,
        "postal_code": str
    }
)

df_types_fixed

,student_id,postal_code,score,date
0,001,010001,85.0,2026-05-26
1,002,020002,90.0,2026-05-27
2,003,030003,NaN,2026-05-28


In [18]:
df_types_fixed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   student_id   3 non-null      object 
 1   postal_code  3 non-null      object 
 2   score        2 non-null      float64
 3   date         3 non-null      object 
dtypes: float64(1), object(3)
memory usage: 228.0+ bytes


## 10. Parsing Dates

Dates in CSV are often read as strings.

`parse_dates` is not a separate function to import.

It is an option inside `pd.read_csv()` that tells pandas which column should be treated as a date.

In [19]:
df_dates = pd.read_csv(
    DATA_TYPE_FILE,
    dtype={
        "student_id": str,
        "postal_code": str
    },
    parse_dates=["date"]
)

df_dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   student_id   3 non-null      object        
 1   postal_code  3 non-null      object        
 2   score        2 non-null      float64       
 3   date         3 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 228.0+ bytes


In [20]:
df_dates

,student_id,postal_code,score,date
0,001,010001,85.0,2026-05-26
1,002,020002,90.0,2026-05-27
2,003,030003,NaN,2026-05-28


## 11. Commas Inside Text

CSV uses commas to separate columns.

But what if the text itself contains a comma?

Example:

```csv
id,comment
1,"Good location, near MRT"
2,"Heavy rain, low visibility"
```

The text with comma should be quoted.

In [21]:
csv_text = '''id,comment
1,"Good location, near MRT"
2,"Heavy rain, low visibility"
3,"No issue"
'''

with open(COMMA_INSIDE_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write(csv_text)

df_comments = pd.read_csv(COMMA_INSIDE_TEXT_FILE)
df_comments

,id,comment
0,1,"Good location, near MRT"
1,2,"Heavy rain, low visibility"
2,3,No issue


### Note

CSV looks simple, but commas and quotes can create parsing problems.

This is why we should use proper libraries such as pandas or Python's built-in `csv` library instead of manually splitting rows using `line.split(",")`.

## 12. Interview-Style Questions



### Question 1

**Why should we avoid manually splitting CSV lines using `split(",")`?**

Suggested answer:

Because values may contain commas inside quoted text. Proper CSV libraries can handle quotes, commas, and escaping correctly.



## 13. Summary

In this notebook, we learned:

1. CSV is a simple text format for tabular data.
2. CSV files usually contain headers, rows, columns, and delimiters.
3. API JSON responses often need to be flattened before saving as CSV.
4. pandas provides convenient functions: `pd.read_csv()` and `df.to_csv()`.
5. CSV files may have issues with missing values, data types, commas in text, and encoding.
6. CSV is useful for simple data exchange, but Parquet is often better for large analytical pipelines.